# L83 · 流水线集成 — 语音到语言

**目标**：串联 Aurora 所有月份成果——实时录音 → Whisper 转写 → Podcast Agent 回答，建立对系统集成和延迟分布的直觉。

🔗 Aurora 连接：`aurora.audio.io`（录音/读写）、`aurora.speech.whisper`（ASR）、`aurora.rag.retriever`（检索）、`aurora.llm.agent`（生成）

这节课的核心问题是：**把各个独立模块拼成一条链时，瓶颈在哪里？**
端到端延迟不是各阶段最大值，而是各阶段之和；任何一环的失败都会让整条链崩溃。
因此需要在每个阶段加计时、加 fallback，而不是假设一切正常。

In [ ]:
import time
import numpy as np
from aurora.audio.io import read_wav

## 1. 流水线延迟分布

端到端延迟由四段串行耗时叠加：

```
总延迟 = T_record + T_asr + T_rag + T_llm
       ≈    0ms   + 500ms +  50ms + 2000ms
       ≈ 2550ms
```

其中 `T_record` 是用户说话的时长（通常 1-5 秒，但录音本身不算延迟——用户已经在说了）。实际感知延迟从用户**停止说话**那一刻开始计算。
`T_asr` 受音频长度和模型规模影响；`T_rag` 受索引大小影响；`T_llm` 是主要瓶颈。

优化方向：让 ASR 和 RAG 并发（`asyncio`），让 LLM 流式输出（减少**首 token 等待**）。

In [ ]:
# 模拟各阶段耗时（秒）
stages = {
    'record':  0.0,    # 录音本身不算等待
    'asr':     0.50,
    'rag':     0.05,
    'llm':     2.00,
}

total = sum(stages.values())
print(f'各阶段耗时: {stages}')
print(f'端到端延迟: {total*1000:.0f}ms')

# 可视化耗时占比
import matplotlib.pyplot as plt
labels = list(stages.keys())
values = list(stages.values())
fig, ax = plt.subplots(figsize=(7, 2))
left = 0
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for label, val, color in zip(labels, values, colors):
    if val > 0:
        ax.barh(0, val, left=left, color=color, edgecolor='white', height=0.4, label=label)
        ax.text(left + val/2, 0, f'{label}\n{val*1000:.0f}ms',
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        left += val
ax.set_xlim(0, total * 1.05)
ax.set_yticks([])
ax.set_xlabel('时间 (秒)')
ax.set_title('端到端延迟分布（感知延迟从用户停止说话后开始）')
plt.tight_layout()
plt.savefig('pipeline_latency.png', dpi=120)
plt.show()
print('✅ 延迟分布图已保存')

## 2. 错误处理与 Fallback 机制

每个阶段都有独立的失败模式：

| 阶段 | 常见失败 | Fallback |
|------|----------|----------|
| 录音 | 静音/噪声 | 检测 RMS < 阈值，提示重说 |
| ASR | 空转写 | 返回空字符串，跳过后续 |
| RAG | 无匹配文档 | 相似度 < 阈值时，返回通用回答 |
| LLM | 超时/OOM | 设 `timeout=5s`，失败时返回缓存答案 |

关键设计原则：**每一阶段返回 `(result, ok, error_msg)` 三元组**，上层用 `ok` 判断是否继续。不要用异常控制流水线走向——异常是程序错误，空录音是正常输入。

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class StageResult:
    value: Any
    ok: bool
    error: str = ''


def check_audio(audio: np.ndarray, sr: int = 16000,
                rms_threshold: float = 0.01) -> StageResult:
    """录音质量检查：RMS 低于阈值判定为静音。"""
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < rms_threshold:
        return StageResult(None, False, f'静音（RMS={rms:.4f} < {rms_threshold}）')
    return StageResult(audio, True)


# 演示：正常音频 vs 静音
audio_ok  = np.random.randn(16000) * 0.1   # 正常说话
audio_sil = np.zeros(16000)                 # 完全静音

r1 = check_audio(audio_ok)
r2 = check_audio(audio_sil)
print(f'正常音频: ok={r1.ok}')
print(f'静音音频: ok={r2.ok}, error="{r2.error}"')
assert r1.ok and not r2.ok
print('✅ 静音检测正确')

## 3. 异步流式输出

LLM 生成通常需要 2-5 秒，但**首 token** 往往在 200ms 内到达。流式输出（streaming）让用户立刻看到回答开头，感知延迟从 2s 降到 0.2s。

实现方式：LLM 返回一个生成器（generator），每次 `yield` 一个 token，调用方边消费边打印：

```python
for token in llm.stream(prompt):
    print(token, end='', flush=True)
```

在 Aurora 中，`aurora.llm.agent.stream_answer(query, context)` 封装此逻辑。
注意：流式和批量接口的**最终输出内容完全相同**，区别只在于何时可见。

In [ ]:
import time

def mock_llm_stream(prompt: str, tokens_per_sec: float = 20.0):
    """模拟 LLM 流式输出，每个 token 间隔 1/tokens_per_sec 秒。"""
    response = '傅里叶变换将时域信号分解为频率成分，是 STFT 和 MFCC 的数学基础。'
    tokens = response  # 逐字符 yield 模拟 token
    for char in tokens:
        time.sleep(1.0 / tokens_per_sec)
        yield char


# 演示流式输出
print('LLM 流式输出（模拟 20 tokens/s）：')
print('─' * 40)
t0 = time.time()
full_response = []
for token in mock_llm_stream('什么是傅里叶变换？'):
    print(token, end='', flush=True)
    full_response.append(token)
print()
print('─' * 40)
elapsed = time.time() - t0
print(f'总耗时: {elapsed:.2f}s，共 {len(full_response)} 个字符')
print('✅ 流式输出演示完成')

## 参数实验：端到端计时

录制5段语音问答，测量并记录每阶段耗时，识别最慢瓶颈。

**参数说明**：
- `n_trials = 5`：实验轮数，越多结果越稳定
- `rms_threshold = 0.01`：静音判定阈值，调高会拒绝更多低音量输入
- `asr_timeout = 3.0`：ASR 超时（秒），超过则用 fallback 空字符串
- `llm_timeout = 5.0`：LLM 超时（秒），超过则返回缓存答案

**预期现象**：LLM 耗时占总延迟 70-80%；启用流式后感知延迟下降明显但总耗时不变。

In [ ]:
import random

def simulate_pipeline(rms_threshold=0.01, asr_delay=0.5,
                      rag_delay=0.05, llm_delay=2.0):
    """模拟完整流水线，返回各阶段耗时字典。"""
    timings = {}

    # 阶段1：录音质量检查
    t0 = time.perf_counter()
    audio = np.random.randn(16000) * 0.1
    result = check_audio(audio, rms_threshold=rms_threshold)
    timings['audio_check'] = time.perf_counter() - t0
    if not result.ok:
        return timings, result.error

    # 阶段2：ASR 转写（模拟）
    t0 = time.perf_counter()
    time.sleep(asr_delay + random.uniform(-0.05, 0.05))
    transcript = '什么是梅尔频率倒谱系数？'
    timings['asr'] = time.perf_counter() - t0

    # 阶段3：RAG 检索（模拟）
    t0 = time.perf_counter()
    time.sleep(rag_delay + random.uniform(-0.01, 0.01))
    context = 'MFCC 基于梅尔滤波器组和 DCT 提取声学特征...'
    timings['rag'] = time.perf_counter() - t0

    # 阶段4：LLM 生成（模拟）
    t0 = time.perf_counter()
    time.sleep(llm_delay + random.uniform(-0.1, 0.1))
    timings['llm'] = time.perf_counter() - t0

    return timings, 'ok'


# ✏️ TODO: 把 n_trials 改成 5，观察每次各阶段耗时的波动
n_trials = 3  # ✏️ TODO
all_timings = []

print(f'运行 {n_trials} 次端到端流水线...')
for i in range(n_trials):
    timings, status = simulate_pipeline()
    total = sum(timings.values())
    all_timings.append(timings)
    print(f'  试验 {i+1}: asr={timings["asr"]*1000:.0f}ms  '
          f'rag={timings["rag"]*1000:.0f}ms  '
          f'llm={timings["llm"]*1000:.0f}ms  '
          f'total={total*1000:.0f}ms')

# 统计瓶颈
avg = {k: np.mean([t[k] for t in all_timings]) for k in all_timings[0]}
bottleneck = max(avg, key=avg.get)
print(f'\n平均各阶段耗时: { {k: f"{v*1000:.0f}ms" for k, v in avg.items()} }')
print(f'最慢瓶颈: {bottleneck}（{avg[bottleneck]*1000:.0f}ms）')
assert bottleneck == 'llm', '预期 LLM 是主要瓶颈'
print('✅ 瓶颈分析完成')

## 本课收束

`check_audio` 守住入口质量，`StageResult` 三元组让每阶段失败可观测；四段串行耗时之和构成端到端延迟，LLM 生成占比约 80%，是首要优化目标。本节完整串联了 `aurora.audio.io`、`aurora.speech.whisper`、`aurora.rag.retriever`、`aurora.llm.agent` 四个模块，形成 Aurora v1 系统闭环。下一节（M6-I2）将引入 `asyncio` 并发，把 ASR 和 RAG 改为并行执行，进一步压低感知延迟。